# 1. What you'll learn

By the end of this notebook, you will be able to:

- describe bias as systematic error and variance as sensitivity to the sample;
- draw the U-shaped expected-error curve from repeated model fits;
- diagnose underfitting, overfitting, and the stabilizing effect of regularization.

**The one question this notebook answers:** Why can a model that fits the
training data more closely predict new data less accurately?

# 2. Setup

This lesson needs repeated random datasets, so reproducibility matters twice:
the main dataset and every resampling experiment derive from seed 0. We use
polynomial regression because one integer—the degree—creates a visible ladder
from rigid straight line to very flexible curve. The printed versions make
the numerical experiment traceable.

In [ ]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

SEED = 0
NOISE_SD = 0.30
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The input `x` lies between -1 and 1. The true target is a smooth wave plus a
slight linear trend: `sin(2πx) + 0.3x`. Observations add Gaussian noise with
standard deviation 0.30. Because the true function is known, we can measure
how far the average fitted model misses truth—the bias—rather than guessing.

For example, if truth is 1.0 and an observation is 1.2, the noise on that row
is +0.2. A good model should recover the underlying 1.0 pattern, not memorize
the accidental +0.2.

In [ ]:
def truth_function(x):
    x = np.asarray(x)
    return np.sin(2 * np.pi * x) + 0.3 * x

rng = np.random.default_rng(SEED)
x = rng.uniform(-1, 1, 180)
true_y = truth_function(x)
observed_y = true_y + rng.normal(0, NOISE_SD, len(x))
frame = pd.DataFrame({"x": x, "target": observed_y, "truth": true_y})
frame["noise"] = frame["target"] - frame["truth"]

print("Shape:", frame.shape)
print("Target task: predict a continuous value from one input")
print("Known noise standard deviation:", NOISE_SD)
print("\nFirst five rows:\n", frame.head().round(3))

# 4. Explore

First, plot observations against known truth. The vertical gap between a dot
and the curve is irreducible noise for that observation. A degree-1 model can
only draw a line through this picture, so it will have high bias. A very high
degree can wiggle through dots, so it risks high variance.

In [ ]:
grid = np.linspace(-1, 1, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(frame["x"], frame["target"], s=22, alpha=0.55, label="observations")
ax.plot(grid, truth_function(grid), color="#176b66", lw=3, label="known truth")
ax.set(title="Noisy observations around a smooth truth", xlabel="x", ylabel="target")
ax.legend()
plt.tight_layout()
plt.show()

**Takeaway:** Noise makes perfect prediction impossible, but the smooth
underlying structure is visible. Capacity controls how much wiggle a model
is allowed to attribute to structure.

Next, check coverage of the input range. Sparse regions give flexible models
room to swing because few observations constrain them. This is a data reason
for variance, not merely a property of an algorithm.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(frame["x"], bins=16, color="#176b66", ax=ax)
ax.set(title="Training evidence is finite across the input range", xlabel="x", ylabel="Rows per bin")
plt.tight_layout()
plt.show()

**Takeaway:** Some intervals contain fewer observations. A high-capacity
curve can change sharply there without paying much training-error penalty.

Because this dataset is synthetic, we can inspect the noise directly. The
model never receives the `truth` or `noise` columns; they exist only so this
lesson can separate model error from randomness.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(frame["noise"], bins=18, kde=True, color="#d38b45", ax=ax)
ax.axvline(0, color="#24302f", ls="--")
ax.set(title="The added noise is centered near zero", xlabel="observed target − truth", ylabel="Rows")
plt.tight_layout()
plt.show()

**Takeaway:** The noise has no stable direction to learn. Chasing its bumps
lowers training error but makes predictions depend strongly on which sample
happened to be drawn.

# 5. Prepare

We make the 60/20/20 split before fitting polynomial transformations. A
polynomial transformer itself has no learned statistics, but the scaler that
follows it does. Keeping both inside the pipeline guarantees that only
training rows determine scale.

The test set remains closed while degree is chosen on validation. Otherwise,
testing degree 1, 2, 3, … on test would turn the holdout into a model-selection
tool and make its winning error optimistic.

In [ ]:
development, test = train_test_split(frame, test_size=0.20, random_state=SEED)
train, validation = train_test_split(development, test_size=0.25, random_state=SEED)

X_train, y_train = train[["x"]], train["target"]
X_validation, y_validation = validation[["x"]], validation["target"]
X_test, y_test = test[["x"]], test["target"]

for name, X_part in [("train", X_train), ("validation", X_validation), ("test", X_test)]:
    print(f"{name:10s} shape={X_part.shape}, x range=({X_part.x.min():.2f}, {X_part.x.max():.2f})")

# 6. Train

Polynomial degree is our capacity knob. Degree 1 is a straight line; degree
2 adds one bend; degree 15 can bend repeatedly. We use tiny ridge
regularization (`alpha=1e-6`) to keep the numerical solve stable without
hiding the capacity effect. Each candidate sees identical training rows.

In [ ]:
def polynomial_model(degree, alpha=1e-6):
    return Pipeline([
        ("polynomial", PolynomialFeatures(degree=degree, include_bias=False)),
        # Scaling prevents high powers such as x^15 from setting the numerical units.
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=alpha)),
    ])

candidate_degrees = [1, 2, 3, 5, 7, 10, 15]
fitted = {}
rows = []
for degree in candidate_degrees:
    model = polynomial_model(degree)
    model.fit(X_train, y_train)
    fitted[degree] = model
    rows.append({
        "degree": degree,
        "train_rmse": mean_squared_error(y_train, model.predict(X_train)) ** 0.5,
        "validation_rmse": mean_squared_error(y_validation, model.predict(X_validation)) ** 0.5,
    })

capacity_results = pd.DataFrame(rows).set_index("degree")
print(capacity_results.round(3))

A picture of three candidates makes “capacity” concrete. The line misses the
repeating shape. A middle degree follows the stable pattern. The highest
degree has enough freedom to react to individual dots, especially near the
boundaries where evidence is thin.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(X_train["x"], y_train, s=18, alpha=0.35, color="#6f7775", label="training rows")
ax.plot(grid, truth_function(grid), color="#172321", lw=3, label="truth")
for degree, color in [(1, "#d38b45"), (5, "#176b66"), (15, "#76558f")]:
    ax.plot(grid, fitted[degree].predict(pd.DataFrame({"x": grid})), lw=2, color=color, label=f"degree {degree}")
ax.set_ylim(-2.2, 2.2)
ax.set(title="Increasing degree increases the shapes a model can draw", xlabel="x", ylabel="prediction")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

# 7. Evaluate

One split can make the curve noisy, so we now repeat the experiment. For each
degree, we draw 40 new training sets, fit 40 models, and predict the same grid.
Bias squared measures how far the *average* prediction is from truth. Variance
measures how much predictions change across samples. Adding the known noise
variance gives expected squared error.

This is an empirical decomposition: every point in the figure comes from
actual fitted models, not a decorative textbook curve.

In [ ]:
decomposition_rng = np.random.default_rng(SEED)
decomposition_rows = []
degree_range = range(1, 16)
repetitions = 40
evaluation_grid = np.linspace(-1, 1, 160)
grid_frame = pd.DataFrame({"x": evaluation_grid})
grid_truth = truth_function(evaluation_grid)

for degree in degree_range:
    predictions = []
    for _ in range(repetitions):
        sample_x = decomposition_rng.uniform(-1, 1, 35)
        sample_y = truth_function(sample_x) + decomposition_rng.normal(0, NOISE_SD, len(sample_x))
        model = polynomial_model(degree, alpha=1e-6)
        model.fit(pd.DataFrame({"x": sample_x}), sample_y)
        predictions.append(model.predict(grid_frame))
    prediction_matrix = np.asarray(predictions)
    mean_prediction = prediction_matrix.mean(axis=0)
    bias_squared = np.mean((mean_prediction - grid_truth) ** 2)
    variance = np.mean(np.var(prediction_matrix, axis=0))
    decomposition_rows.append({
        "degree": degree,
        "bias_squared": bias_squared,
        "variance": variance,
        "expected_error": bias_squared + variance + NOISE_SD ** 2,
    })

decomposition = pd.DataFrame(decomposition_rows).set_index("degree")
best_expected_degree = int(decomposition["expected_error"].idxmin())
print(decomposition.round(4))
print("Lowest repeated expected error at degree:", best_expected_degree)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(decomposition.index, decomposition["bias_squared"], marker="o", label="bias²", color="#d38b45")
ax.plot(decomposition.index, decomposition["variance"], marker="o", label="variance", color="#76558f")
ax.plot(decomposition.index, decomposition["expected_error"], marker="o", lw=3, label="bias² + variance + noise", color="#176b66")
ax.axvline(best_expected_degree, ls="--", color="#6f7775", label=f"minimum: degree {best_expected_degree}")
ax.set_yscale("log")
ax.set(title="The U-shaped error curve measured from 600 fitted models", xlabel="Polynomial degree (capacity)", ylabel="Mean squared contribution — log scale")
ax.legend()
plt.tight_layout()
plt.savefig(Path.cwd() / "key_figure.png", dpi=160, bbox_inches="tight")
plt.show()

**Takeaway:** Bias falls as the model gains useful flexibility, while variance
eventually rises because high-degree curves react to sample noise. Their sum
has a measured minimum between the two extremes.

We still honor the real workflow: select degree using the single validation
set, then compare that frozen choice with a mean baseline on untouched test.

In [ ]:
selected_degree = int(capacity_results["validation_rmse"].idxmin())
selected_model = fitted[selected_degree]
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

selected_test_rmse = mean_squared_error(y_test, selected_model.predict(X_test)) ** 0.5
baseline_test_rmse = mean_squared_error(y_test, baseline.predict(X_test)) ** 0.5
print(f"Validation-selected degree: {selected_degree}")
print(f"Selected model test RMSE: {selected_test_rmse:.3f}")
print(f"Mean baseline test RMSE: {baseline_test_rmse:.3f}")

# 8. Break it

We now give a degree-15 unregularized model only 12 training points. It has
more freedom than the data can constrain, so it can nearly memorize those
points while swinging wildly between them. A ridge penalty makes large,
cancelling coefficients expensive and should reduce the test damage.

In [ ]:
tiny = train.sample(12, random_state=SEED).sort_values("x")
unregularized = Pipeline([
    ("polynomial", PolynomialFeatures(degree=15, include_bias=False)),
    ("model", LinearRegression()),
]).fit(tiny[["x"]], tiny["target"])
stabilized = polynomial_model(degree=15, alpha=0.1).fit(tiny[["x"]], tiny["target"])

broken_rows = []
for name, model in [("degree 15, no penalty", unregularized), ("degree 15, ridge", stabilized)]:
    broken_rows.append({
        "model": name,
        "tiny_train_rmse": mean_squared_error(tiny["target"], model.predict(tiny[["x"]])) ** 0.5,
        "test_rmse": mean_squared_error(y_test, model.predict(X_test)) ** 0.5,
        "max_abs_grid_prediction": np.max(np.abs(model.predict(grid_frame))),
    })
broken = pd.DataFrame(broken_rows).set_index("model")
print(broken.round(3))
print("\nDamage ratio (unregularized test / selected test):", round(broken.iloc[0]["test_rmse"] / selected_test_rmse, 1))

**Use this when…** you need to reason about model capacity, decide whether
more data or a simpler model may help, or explain why training performance
and new-data performance move in different directions.

**Don't use this when…** you are treating “bias” as social or measurement
bias. Those are important but different concepts; the statistical
bias–variance decomposition does not audit fairness or data provenance.

## Try this

1. Increase `NOISE_SD` from 0.30 to 0.60. The irreducible floor rises and the
   best degree may move lower because fine structure becomes harder to justify.
2. Change repeated-sample size from 35 to 100. Variance should shrink,
   especially for higher degrees; more evidence supports more capacity.
3. Change ridge `alpha` in `polynomial_model` from `1e-6` to `0.1`. Bias should
   rise slightly and high-degree variance should fall. Look for a flatter U.